# Candidate-Method Feasibility Assessment

**Purpose:** Evaluate the candidate feature-extraction methods NOT implemented in Phase 0, so the Feature Exploration cycle (Phase 0.5) starts with a clear, prioritised agenda.

**Motivation:** The Phase-0 alignment-free baseline only weakly reproduced typology:
- `class_separation` ≈ +1.79 (positive direction but tiny vs. literature's 15–30 nPVI-unit gap)
- `spearman_against_reference` ≈ −0.30 (rank ordering partially inverted vs. Grabe & Low 2002)

The four methods assessed here are:
1. **Pretrained SSL embeddings** — executable probe (priority)
2. **Forced-alignment metrics via MFA** — documented assessment (no install)
3. **Learned envelope embedding (Deloche-style)** — documented assessment (no training)
4. **Comparison table** and agenda

---

## 1. Pretrained SSL Embeddings — Executable Probe

### What are SSL embeddings?

Self-supervised learning (SSL) models such as **wav2vec 2.0** (Baevski et al. 2020), **HuBERT** (Hsu et al. 2021), and **XLS-R** (Babu et al. 2021) are transformer encoders pretrained on raw audio via masked-prediction objectives — no labels, no transcripts. The **last hidden state** (a sequence of 768-d or 1024-d frame-level vectors) encodes rich acoustic-phonetic representations. Mean-pooling these vectors over the clip yields a single fixed-length embedding per clip.

For language identification, **VoxLingua107 ECAPA-TDNN** (Valk & Alumäe 2021) is a speaker-/language-discriminative embedding trained on 107 languages and achieves >93% top-1 accuracy on VoxLingua107's test set. XLS-R (300 M parameter multilingual wav2vec) was pretrained on 436k hours across 128 languages.

### Why this is the priority

- **Inference-only** — no training pipeline needed; models run on CPU.
- **Multilingual coverage** — XLS-R and VoxLingua107 ECAPA cover all 8 seed languages.
- **One extra hop to `proximity`** — mean-pool the hidden state → 768-d vector → plug into the existing cosine / Ward distance infrastructure.
- **Already available** — `facebook/wav2vec2-base` is cached locally; `transformers 5.12.1` is installed in the venv.

### Probe

In [1]:
import glob
import numpy as np
import torch
from pathlib import Path
from scipy.spatial.distance import cosine as cosine_distance
from transformers import AutoFeatureExtractor, AutoModel

from musiclang.config import DATA_DIR
from musiclang.audio import load_audio

# ── 1. Discover clips (two distinct languages) ──────────────────────────────
all_clips = glob.glob(str(DATA_DIR / "clips" / "*" / "*.wav"))

# Group by language (parent folder name)
from collections import defaultdict
by_lang: dict[str, list[str]] = defaultdict(list)
for p in all_clips:
    lang = Path(p).parent.name
    by_lang[lang].append(p)

langs_with_clips = sorted(by_lang.keys())
print(f"Languages with clips: {langs_with_clips}")

# Pick the first clip from the first two distinct languages
lang_a, lang_b = langs_with_clips[0], langs_with_clips[1]
clip_a = sorted(by_lang[lang_a])[0]
clip_b = sorted(by_lang[lang_b])[0]
print(f"Clip A: {clip_a}  ({lang_a})")
print(f"Clip B: {clip_b}  ({lang_b})")

# ── 2. Load and truncate to ~15 s ───────────────────────────────────────────
SR = 16_000
MAX_SAMPLES = SR * 15

audio_a = load_audio(clip_a, sr=SR)[:MAX_SAMPLES]
audio_b = load_audio(clip_b, sr=SR)[:MAX_SAMPLES]
print(f"\nAudio A duration: {len(audio_a)/SR:.1f} s ({len(audio_a)} samples)")
print(f"Audio B duration: {len(audio_b)/SR:.1f} s ({len(audio_b)} samples)")

# ── 3. Load facebook/wav2vec2-base (cached locally) ────────────────────────
MODEL_ID = "facebook/wav2vec2-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(MODEL_ID)
model.eval()
print(f"\nModel loaded: {MODEL_ID}")

# ── 4. Embed: feature-extract → forward pass → mean-pool ───────────────────
def embed_clip(audio: np.ndarray) -> np.ndarray:
    """Return a 768-d mean-pooled embedding for one audio clip."""
    inputs = feature_extractor(
        audio, sampling_rate=SR, return_tensors="pt", padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)
    # last_hidden_state: (1, T, 768) → mean over T → (768,)
    emb = outputs.last_hidden_state.squeeze(0).mean(dim=0).numpy()
    return emb

emb_a = embed_clip(audio_a)
emb_b = embed_clip(audio_b)

# ── 5. Report ───────────────────────────────────────────────────────────────
print(f"\nEmbedding shape (A — {lang_a}): {emb_a.shape}")
print(f"Embedding shape (B — {lang_b}): {emb_b.shape}")

dist = float(cosine_distance(emb_a, emb_b))
print(f"\nCosine distance ({lang_a} <-> {lang_b}): {dist:.4f}")
print("(Reference: english <-> finnish ≈ 0.12 in controller smoke-test)")

Languages with clips: ['english', 'finnish', 'french', 'german', 'greek', 'italian', 'polish', 'spanish']
Clip A: C:\Users\nikol\Dev\Personal\Linguistics\data\clips\english\00.wav  (english)
Clip B: C:\Users\nikol\Dev\Personal\Linguistics\data\clips\finnish\00.wav  (finnish)



Audio A duration: 15.0 s (240000 samples)
Audio B duration: 15.0 s (240000 samples)


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model loaded: facebook/wav2vec2-base



Embedding shape (A — english): (768,)
Embedding shape (B — finnish): (768,)

Cosine distance (english <-> finnish): 0.1193
(Reference: english <-> finnish ≈ 0.12 in controller smoke-test)


### Conclusion — SSL embeddings

The probe confirms:
- The pooled embedding is a 768-d vector (one per clip), as expected from `facebook/wav2vec2-base`.
- The cosine distance between two different-language clips is non-trivial and measurable — the embedding space captures cross-language variation.

**SSL embeddings are the recommended FIRST method for the Feature Exploration cycle** because:
1. Inference-only — no training pipeline, no labelled data, runs on CPU today.
2. The upgrade path is straightforward: swap to `facebook/wav2vec2-large-xlsr-53` (300 M, 53 languages) or `facebook/wav2vec2-xls-r-300m` (128 languages) for stronger multilingual representations.
3. VoxLingua107 ECAPA-TDNN (`speechbrain/lang-id-voxlingua107-ecapa`) is a drop-in alternative specifically fine-tuned for language identification on 107 languages.
4. The resulting 768-d (or 1024-d) per-clip embeddings feed the existing `proximity.distance` module without modification — just replace the feature DataFrame.

---

## 2. Forced-Alignment Metrics (MFA) — Documented Assessment

> **No MFA install or `mfa` commands are run in this notebook.** This is a reasoned assessment based on publicly documented MFA capabilities and our data constraints.

### What MFA provides

[Montreal Forced Aligner](https://montreal-forced-aligner.readthedocs.io) (McAuliffe et al. 2017) provides phone-level forced alignment given:
- an audio file,
- a **verbatim text transcript**, and
- a pronunciation dictionary + acoustic model for the target language.

With phone boundaries, classic rhythm metrics become exact: %V, nPVI-V, rPVI-C, VarcoC, delta-C / delta-V — all computed on verified phone intervals rather than VAD-estimated syllable approximations. This is the gold-standard pipeline used by Grabe & Low (2002) and most subsequent typological work.

### Pretrained model availability for the 8 seed languages

The [MFA model repository](https://mfa-models.readthedocs.io) provides pretrained acoustic models and pronunciation dictionaries for:

| Language | Acoustic model | Pronunciation dictionary |
|----------|---------------|--------------------------|
| English  | Yes (multiple, e.g. `english_us_arpa`) | Yes |
| German   | Yes (`german_mfa`) | Yes |
| French   | Yes (`french_mfa`) | Yes |
| Spanish  | Yes (`spanish_mfa`) | Yes |
| Italian  | Yes (`italian_mfa`) | Yes |
| Polish   | Yes (`polish_mfa`) | Yes |
| Greek    | Partial (community models; quality variable) | Partial |
| Finnish  | Partial (community models; quality variable) | Partial |

Coverage is good for the six Western European languages; Greek and Finnish are more variable.

### Windows install cost

MFA on Windows is distributed via **conda/conda-forge** (`conda install -c conda-forge montreal-forced-aligner`). It depends on a compiled Kaldi backend, which is not pip-installable. The install is heavy (~1–2 GB with dependencies including OpenFST, BLAS libraries, and the Kaldi toolchain). It is not compatible with the project's pip venv and would require a separate conda environment.

### CRITICAL feasibility constraint: transcript dependency

**MFA requires a verbatim text transcript aligned to the audio.** Our radio clips are **unlabelled spontaneous speech** — we have no transcripts. Two paths exist to obtain them:

1. **Run ASR first** (e.g., OpenAI Whisper) to produce transcripts, then pass those to MFA. This adds a large dependency (Whisper large-v3: ~1.5 GB) and an ASR error surface: every transcription error propagates to a misalignment, corrupting the phone boundaries MFA was meant to provide.
2. **Source pre-transcribed speech data** (e.g., Common Voice, read speech corpora). This abandons our radio-stream corpus entirely and changes the data distribution.

### Conclusion — MFA

MFA carries **high infrastructure weight** (conda-only, Kaldi, 1–2 GB) and a **transcript dependency** that our unlabelled radio clips cannot satisfy without running ASR first. The marginal accuracy gain over the alignment-free %V/nPVI baseline (which approximates the same C/V-interval metrics without alignment) must be weighed against:
- An ASR error chain on spontaneous radio speech (non-trivial WER, especially for Greek/Finnish).
- The conda/pip environment split complicating CI and reproducibility.
- The fact that SSL embeddings (Section 1) offer a different representational strategy — not just better boundary placement — and are available immediately.

**Recommended position:** MFA is a calibration tool for a future **read-speech sub-corpus** (e.g., a 10-minute Common Voice sample per language), not a primary path for our radio-clip corpus. Prioritise SSL embeddings and envelope-RNN first.

---

## 3. Learned Envelope Embedding (Deloche-style) — Documented Assessment

> **No model training or large downloads are performed in this notebook.** This is a reasoned assessment based on the published Deloche et al. (2024) architecture and our data constraints.

### What the method does

Deloche et al. (2024) (*"Unsupervised language rhythm learning from speech envelopes"*) train a **recurrent neural network** (GRU/LSTM) on sequences of **speech envelope + voicing probability** frames — a ~2-D input at roughly 100 Hz — using a self-supervised objective (e.g., contrastive or next-step prediction). The network learns to encode rhythmic structure into a fixed-length embedding without phonetic labels. The resulting embeddings cluster by rhythm class (stress vs. syllable) in a low-dimensional space.

### Why there is no pretrained checkpoint

As of mid-2025, **no widely available pretrained checkpoint** for the Deloche-style envelope-RNN has been released to a public model hub (HuggingFace, Zenodo, or GitHub releases). The original paper's code is research-grade and requires the user to train from scratch.

### Data and compute requirements for training

Based on the paper's training setup and standard self-supervised audio practice:

| Resource | Rough estimate |
|----------|---------------|
| Clips per language | ~50–200 clips (≥ 30 s each) |
| Total audio | ~5–20 hours across 8 languages |
| Training time | Several hours on a single GPU (or 1–2 days on CPU) |
| Model size | Small (GRU: ~1–5 M parameters); no GPU strictly required but strongly recommended |

Our current corpus has **31 clips across 8 languages** — roughly 4 clips per language on average, with Finnish at 1 clip. This is likely insufficient to train a generalisable envelope-RNN; the self-supervised objective would overfit to station-specific prosodic patterns. Reaching the 50+ clips/language threshold would require expanding the collection pipeline.

### How the embeddings would feed `proximity`

Once trained, the envelope-RNN encoder produces a fixed-length embedding per clip (e.g., 128-d or 256-d), analogous to the SSL embeddings in Section 1. These feed the existing `proximity.distance` module identically: compute the per-language mean embedding, then apply cosine / Ward distance and MDS for visualisation. The representational strategy is different from SSL — it focuses purely on amplitude-envelope rhythm, not phonetic content — which may yield more interpretable axes (slower envelope modulation = stress-timed, flatter = syllable-timed).

### Conclusion — envelope-RNN

The envelope-RNN approach is **conceptually attractive** (interpretable rhythm-focused representation, lightweight input features) but **requires a training pipeline and substantially more data** than we currently have (~50–200 clips/language vs. our current ~4). There is no pretrained checkpoint available today.

**Recommended position:** Envelope-RNN is a medium-term investment. It becomes viable after (a) the collection pipeline scales to 50+ clips/language, and (b) the SSL embeddings have been evaluated. It is the better choice over MFA for our unlabelled corpus because it requires no transcripts, but it needs training infra and data the project doesn't yet have.

---

## 4. Comparison Table

| Method | Interpretability | Infra cost | Expected fidelity | Blog-friendliness |
|--------|-----------------|------------|-------------------|-----------------|
| **Alignment-free baseline** (%V, nPVI, rPVI, VarcoC) | High — metrics have known phonetic meaning | Minimal (pure Python/librosa) | Weak (class_sep ≈ +1.79, Spearman ≈ −0.30 vs. literature) | High — directly maps to textbook rhythm typology |
| **SSL embeddings** (wav2vec2 / XLS-R / VoxLingua107) | Low–Medium — latent space, but UMAP/PCA can reveal structure | Low (pip install, inference-only, CPU-capable) | Medium–High — multilingual SSL models encode cross-language phonetic variation | High — "plug in a HuggingFace model" is a compelling narrative |
| **MFA metrics** (forced-alignment %V/nPVI) | High — exact phone-boundary measurements | High (conda/Kaldi, transcript dependency, ASR chain for radio clips) | High in principle — gold-standard for typological work; degrades with ASR errors | Medium — requires explaining forced alignment and ASR pipeline |
| **Envelope-RNN** (Deloche-style) | Medium — rhythm-focused but learned representation | Medium (training pipeline, 50+ clips/language, GPU recommended) | Medium–High — focuses on envelope rhythm, potentially cleaner than SSL for typology | Medium-High — "we trained a custom RNN on speech rhythm" is blog-worthy |

---

## 5. Agenda for the Feature Exploration Cycle (Phase 0.5)

### Priority order

1. **SSL embeddings — FIRST** (this cycle)
   - Load `facebook/wav2vec2-xls-r-300m` or `speechbrain/lang-id-voxlingua107-ecapa`.
   - Embed all 31 current clips; compute per-language mean embeddings.
   - Re-run `class_separation` and `spearman_against_reference` on the SSL-derived distances.
   - Compare dendrogram and MDS vs. the alignment-free baseline.
   - **Cost:** one afternoon; no training; no additional data.

2. **Expand clip collection** (enabling envelope-RNN later)
   - Target 50+ clips per language to make envelope-RNN training viable.
   - Also improves reliability of the alignment-free baseline (reduces `*_std` noise).

3. **Envelope-RNN — SECOND** (next cycle, after data expansion)
   - Implement the Deloche-style encoder once 50+ clips/language are available.
   - Evaluate whether envelope-rhythm embeddings separate rhythm classes more cleanly than SSL.

4. **MFA metrics — OPTIONAL, read-speech only**
   - Applicable only to a pre-transcribed sub-corpus (e.g., Common Voice samples).
   - Treat as a calibration benchmark, not a primary feature-extraction path for radio clips.

### Rationale

The alignment-free baseline gave a weak positive signal with negative Spearman rank correlation. SSL embeddings are the cheapest next step that could qualitatively change the representational fidelity — moving from hand-crafted rhythm metrics to learned acoustic representations. Envelope-RNN is the right long-term method for interpretable, rhythm-focused embeddings, but it requires data we don't yet have. MFA is the right calibration tool for read speech, not for spontaneous radio.

---

*Notebook authored for Phase 0 → Phase 0.5 transition. All SSL measurements are from the executable probe above; MFA and envelope-RNN assessments are documented reasoning, not measurements.*